In [1]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import importlib

import env.trading_env as te
import features.feature_engineering as fe
importlib.reload(te)
importlib.reload(fe)

from stable_baselines3 import PPO

In [2]:
# Read data
df = pd.read_excel('../data/data.xlsx', skiprows=6, header=0)
df.columns = ['date', 'gold', 'silver', 'copper']
df = df[pd.to_datetime(df['date'], errors='coerce').notna()]
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

In [3]:
# Feature engineering
features_final = fe.build_features(df)

# Split
train = features_final[features_final.index <= '2023-12-31']
test = features_final[features_final.index >= '2024-01-01']
price_train = df[df.index.isin(train.index)]
price_test = df[df.index.isin(test.index)]

print(f"Train: {len(train)}days, Test: {len(test)}days")

Train: 2592days, Test: 485days


In [4]:
# Set up the environment
train_env = te.MetalTradingEnv(train, price_train)

# initialize PPO
model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    verbose=1
)

# train
model.learn(total_timesteps=200_000)
print("training finish")

# Save model
model.save("../agents/ppo_metal")
print("model saved")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
-----------------------------
| time/              |      |
|    fps             | 958  |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2048 |
-----------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 2.59e+03   |
|    ep_rew_mean          | -3.13      |
| time/                   |            |
|    fps                  | 624        |
|    iterations           | 2          |
|    time_elapsed         | 6          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.01062146 |
|    clip_fraction        | 0.0957     |
|    clip_range           | 0.2        |
|    entropy_loss         | -5.68      |
|    explained_variance   | -2.64      |
|    learning_rate        | 0.0003     |
|    loss                 | -0.00397   |